#  LangChain의 RAG 콤포넌트 - 벡터 저장소(Vector Store) 

### **학습 목표:**  임베딩 모델과 벡터 데이터베이스를 효과적으로 연동할 수 있다

### **사전 준비:**

- 필요 패키지: `langchain-chroma`, `faiss-cpu` (벡터검색에 특화), `langchain-pinecone`
- 환경변수: OPENAI_API_KEY, PINECONE_API_KEY 설정 필요

### **실습 자료**: 해당 없음

---

# 환경 설정 및 준비

`(1) Env 환경변수`

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

`(2) 기본 라이브러리`

In [4]:
import os
from glob import glob

from pprint import pprint
import json

# 벡터 저장소 (Vector Store)

- 개념:
    - 벡터화된 데이터를 효율적으로 저장하고 검색하기 위한 특수 데이터베이스 시스템
    - 텍스트나 이미지 등의 비정형 데이터를 고차원 벡터 공간에 매핑하여 저장
    - 유사도 기반 검색을 통해 의미적으로 가까운 데이터를 빠르게 검색 가능 

- LangChain의 벡터 저장소 종류:
    - **Chroma**: 경량화된 임베딩 데이터베이스로 로컬 개발에 적합
    - **FAISS**: Facebook AI가 개발한 고성능 유사도 검색 라이브러리
    - **Pinecone**: 완전 관리형 벡터 데이터베이스 서비스
    - Milvus: 분산 벡터 데이터베이스로 대규모 데이터 처리에 적합
    - PostgreSQL: pgvector 확장을 통해 벡터 저장 및 검색 기능을 제공

- 주요 기능:
    - 벡터 색인화: 효율적인 검색을 위한 데이터 구조화를 수행
    - 근접 이웃 검색: 주어진 쿼리와 가장 유사한 벡터들을 검색 
    - 메타데이터 관리: 벡터와 관련된 부가 정보를 함께 저장하고 검색

- 사용 사례:
    - 시맨틱 문서 검색: 문서의 의미를 이해하여 검색
    - 추천 시스템: 유사한 아이템을 추천
    - 중복 데이터 감지: 유사한 콘텐츠를 검색 
    - 질의응답 시스템: 관련 문서에서 답변을 생성하는데 필요한 근거를 검색 

- 벡터 저장소 선택 가이드

    | 벡터 저장소 | 장점 | 단점 | 추천 사용 사례 |
    |------------|------|------|---------------|
    | **Chroma** | 설정 간단, 로컬 개발 용이 | 대규모 데이터 처리 제한적 | 프로토타입, 소규모 프로젝트 |
    | **FAISS** | 빠른 검색 속도, 대용량 처리 | 별도 저장소 없음 (메모리/파일만) | 고성능 검색, 대규모 데이터 |
    | **Pinecone** | 완전 관리형, 확장성 우수 | 유료 서비스, API 키 필요 | 프로덕션 환경, 클라우드 배포 |

### 1. **Chroma**

- 사용자 편의성이 우수한 오픈소스 벡터 저장소
- `langchain-chroma` 패키지 설치

`(1) 벡터 저장소 초기화`

In [5]:
# 벡터 저장소에 문서를 저장할 때 적용할 임베딩 모델
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
embeddings_model = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

e:\sw\dev\ai\modu_llm7\faq_bot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 14425.08it/s]


In [ ]:
# from langchain_ollama import OllamaEmbeddings 

# embeddings_model = OllamaEmbeddings(model="dengcao/Qwen3-Embedding-0.6B:Q8_0")

In [ ]:
# 벡터 저장소 생성
from langchain_chroma import Chroma

chroma_db = Chroma(
    collection_name="ai_sample_collection", # RDBMS로 치면  테이블
    embedding_function=embeddings_model,    # 필요한 임베딩 모델을 설정 
    persist_directory="./chroma_db",        # 저장할 폴더 경로
)

In [7]:
# 현재 저장된 컬렉션 데이터 확인
chroma_db.get()

{'ids': [],
 'embeddings': None,
 'documents': [],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': []}

`(2) 벡터 저장소 관리`  

- 문서 추가: `vector_store.add_documents(documents, ids)`

In [ ]:
from langchain_core.documents import Document
# 현재 저장된 컬렉션 데이터 확인
chroma_db.get()
# 문서 데이터 - (텍스트, 소스)
documents = [
    ("인공지능은 컴퓨터 과학의 한 분야입니다.", "AI 개론"),
    ("머신러닝은 인공지능의 하위 분야입니다.", "AI 개론"),
    ("딥러닝은 머신러닝의 한 종류입니다.", "딥러닝 입문"),
    ("자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.", "AI 개론"),
    ("컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.", "딥러닝 입문")
]

# Document 객체 생성
doc_objects = []
for content, source in documents:
    doc = Document(
        page_content=content,
        metadata={"source": source},
    )
    doc_objects.append(doc)


# 순차적 ID 리스트 생성
doc_ids = [f"DOC_{i}" for i in range(1, len(doc_objects) + 1)]

# 문서를 벡터 저장소에 저장
added_doc_ids = chroma_db.add_documents(documents=doc_objects, ids=doc_ids)  # 문서 객체와 문서 id 입력 

# 벡터 저장소에 저장된 문서를 확인
print(f"{len(added_doc_ids)}개의 문서가 성공적으로 벡터 저장소에 추가되었습니다.")
print(added_doc_ids)

5개의 문서가 성공적으로 벡터 저장소에 추가되었습니다.
['DOC_1', 'DOC_2', 'DOC_3', 'DOC_4', 'DOC_5']


In [10]:
# 저장된 문서 검색
query = "인공지능과 머신러닝의 관계는?"
results = chroma_db.similarity_search(query, k=2)

print(f"\n쿼리: {query}")
print("가장 유사한 문서:")
for doc in results:
    print(f"- {doc.page_content} [출처: {doc.metadata['source']}]")


쿼리: 인공지능과 머신러닝의 관계는?
가장 유사한 문서:
- 머신러닝은 인공지능의 하위 분야입니다. [출처: AI 개론]
- 딥러닝은 머신러닝의 한 종류입니다. [출처: 딥러닝 입문]


In [ ]:
# 현재 저장된 컬렉션 데이터 확인
chroma_db.get() # 양이 많을 경우 부하가 생길 수 있어 잘 사용하지 않음 

{'ids': ['DOC_1', 'DOC_2', 'DOC_3', 'DOC_4', 'DOC_5'],
 'embeddings': None,
 'documents': ['인공지능은 컴퓨터 과학의 한 분야입니다.',
  '머신러닝은 인공지능의 하위 분야입니다.',
  '딥러닝은 머신러닝의 한 종류입니다.',
  '자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.',
  '컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'source': 'AI 개론'},
  {'source': 'AI 개론'},
  {'source': '딥러닝 입문'},
  {'source': 'AI 개론'},
  {'source': '딥러닝 입문'}]}

- 문서 수정: `vector_store.update_document(document_id, document)`

In [12]:
# 업데이트할 문서 생성
updated_document_1 = Document(
    page_content="인공지능은 컴퓨터 과학의 핵심 분야 중 하나로, 기계학습과 딥러닝을 포함합니다.",
    metadata={"source": "AI 개론"},
)

updated_document_2 = Document(
    page_content="머신러닝은 데이터로부터 학습하여 예측과 결정을 내리는 인공지능의 하위 분야입니다.",
    metadata={"source": "AI 개론"},
)

updated_document_3 = Document(
    page_content="딥러닝은 머신러닝의 한 종류로, 심층 신경망을 사용하여 학습합니다.",
    metadata={"source": "딥러닝 입문"},
)


# 단일 문서 업데이트
chroma_db.update_document(document_id="DOC_1", document=updated_document_1)

# 여러 문서 한 번에 업데이트
chroma_db.update_documents(
    ids=["DOC_2", "DOC_3"],
    documents=[updated_document_2, updated_document_3]
)

print("문서 업데이트 완료")

문서 업데이트 완료


In [13]:
# 저장된 문서 검색 예시
query = "인공지능과 머신러닝의 관계는?"
results = chroma_db.similarity_search(query, k=2)

print(f"\n쿼리: {query}")
print("가장 유사한 문서:")
for doc in results:
    print(f"- {doc.page_content} [출처: {doc.metadata['source']}]")


쿼리: 인공지능과 머신러닝의 관계는?
가장 유사한 문서:
- 머신러닝은 데이터로부터 학습하여 예측과 결정을 내리는 인공지능의 하위 분야입니다. [출처: AI 개론]
- 인공지능은 컴퓨터 과학의 핵심 분야 중 하나로, 기계학습과 딥러닝을 포함합니다. [출처: AI 개론]


- 문서 삭제: `vector_store.delete(ids)`

In [14]:
# 문서 id를 지정하여 삭제
chroma_db.delete(ids=["DOC_5"])

In [15]:
# 컬렉션 확인
chroma_db.get()

{'ids': ['DOC_1', 'DOC_2', 'DOC_3', 'DOC_4'],
 'embeddings': None,
 'documents': ['인공지능은 컴퓨터 과학의 핵심 분야 중 하나로, 기계학습과 딥러닝을 포함합니다.',
  '머신러닝은 데이터로부터 학습하여 예측과 결정을 내리는 인공지능의 하위 분야입니다.',
  '딥러닝은 머신러닝의 한 종류로, 심층 신경망을 사용하여 학습합니다.',
  '자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'source': 'AI 개론'},
  {'source': 'AI 개론'},
  {'source': '딥러닝 입문'},
  {'source': 'AI 개론'}]}

`(3) 문서 검색`  

- 유사도 검색
    - 주어진 쿼리와 가장 유사한 문서를 반환
    -  k=2는 상위 2개의 결과를 반환하도록 지정
    - filter를 사용하여 특정 출처의 문서만 검색 가능

In [16]:
query = "인공지능과 머신러닝의 차이점은 무엇인가요?"
results = chroma_db.similarity_search(
    query,
    k=2,
    filter={"source": "딥러닝 입문"}
)

print("유사도 검색 결과:")
for doc in results:
    print(f"- {doc.page_content} [출처: {doc.metadata['source']}]")

유사도 검색 결과:
- 딥러닝은 머신러닝의 한 종류로, 심층 신경망을 사용하여 학습합니다. [출처: 딥러닝 입문]


**유사도 점수 해석**

- 유사도 점수가 포함된 검색
    - 유사도 점수를 함께 반환
    - 점수가 낮을수록 더 유사한 것을 의미 (거리 기준으로 점수가 산정되기 때문)


- 검색 방법 비교

    | 메서드 | 반환값 | 점수 범위 | 점수 해석 |
    |--------|--------|-----------|-----------|
    | `similarity_search` | 문서 리스트만 | - | 점수 없음 |
    | `similarity_search_with_score` | (문서, 거리점수) | 0 ~ ∞ | **낮을수록** 유사 |
    | `similarity_search_with_relevance_scores` | (문서, 관련성점수) | 0 ~ 1 | **높을수록** 유사 |



In [17]:
query = "딥러닝은 어떤 분야에서 사용되나요?"
results = chroma_db.similarity_search_with_score(
    query,
    k=2,
    filter={"source": "AI 개론"}
)

print("점수가 포함된 유사도 검색 결과:\n")
for doc, score in results:
    print(f"- 점수: {score:.4f}")
    print(f"  내용: {doc.page_content}")
    print(f"  [출처: {doc.metadata['source']}]")
    print()

점수가 포함된 유사도 검색 결과:

- 점수: 0.7292
  내용: 인공지능은 컴퓨터 과학의 핵심 분야 중 하나로, 기계학습과 딥러닝을 포함합니다.
  [출처: AI 개론]

- 점수: 0.8394
  내용: 머신러닝은 데이터로부터 학습하여 예측과 결정을 내리는 인공지능의 하위 분야입니다.
  [출처: AI 개론]



- 관련성 점수가 포함된 검색
    - 문서와 함께 0에서 1 사이의 관련성 점수를 반환
    - 0은 가장 관련성이 낮고, 1은 가장 관련성이 높음을 의미

In [18]:
query = "딥러닝은 어떤 분야에서 사용되나요?"
results = chroma_db.similarity_search_with_relevance_scores(
    query,
    k=2,
    filter={"source": "AI 개론"}
)

print(f"쿼리: {query}")
print("\n검색 결과 (관련성 점수 포함):")
for doc, score in results:
    print(f"- 관련성 점수: {score:.4f}")
    print(f"  내용: {doc.page_content}")
    print(f"  [출처: {doc.metadata['source']}]")
    print()

쿼리: 딥러닝은 어떤 분야에서 사용되나요?

검색 결과 (관련성 점수 포함):
- 관련성 점수: 0.4844
  내용: 인공지능은 컴퓨터 과학의 핵심 분야 중 하나로, 기계학습과 딥러닝을 포함합니다.
  [출처: AI 개론]

- 관련성 점수: 0.4065
  내용: 머신러닝은 데이터로부터 학습하여 예측과 결정을 내리는 인공지능의 하위 분야입니다.
  [출처: AI 개론]



`(4) 벡터 저장소 로드`  

In [19]:
chroma_db2 = Chroma(
    collection_name="ai_sample_collection",
    embedding_function=embeddings_model,
    persist_directory="./chroma_db",
)

In [20]:
# 미리 임베딩된 쿼리 벡터를 사용하여 검색
query = "딥러닝은 어떤 분야에서 사용되나요?"
results = chroma_db2.similarity_search_with_relevance_scores(
    query,
    k=2,
    filter={"source": "딥러닝 입문"}
)

print(f"쿼리: {query}")
print("\n검색 결과 (관련성 점수 포함):")
for doc, score in results:
    print(f"- 관련성 점수: {score:.4f}")
    print(f"  내용: {doc.page_content}")
    print(f"  [출처: {doc.metadata['source']}]")
    print()

쿼리: 딥러닝은 어떤 분야에서 사용되나요?

검색 결과 (관련성 점수 포함):
- 관련성 점수: 0.5919
  내용: 딥러닝은 머신러닝의 한 종류로, 심층 신경망을 사용하여 학습합니다.
  [출처: 딥러닝 입문]



### **2 FAISS(Facebook AI Similarity Search)**

- 효율적인 벡터 유사도 검색 및 클러스터링을 위한 오픈소스 벡터 저장소 
- `faiss-cpu` 패키지 설치

`(1) 벡터 저장소 초기화`

In [21]:
embeddings_model

HuggingFaceEmbeddings(model_name='BAAI/bge-m3', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [22]:
len(embeddings_model.embed_query("hello world"))

1024

In [23]:
# 벡터 저장소 생성
import faiss 
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

# FAISS 인덱스 초기화 (유클리드 거리 사용)
faiss_index = faiss.IndexFlatL2(len(embeddings_model.embed_query("hello world")))
print("FAISS 인덱스 초기화 완료")

C:\Users\JSPark\AppData\Local\Temp\ipykernel_37856\3740493069.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.docstore.in_memory import InMemoryDocstore


FAISS 인덱스 초기화 완료


In [24]:
# FAISS 벡터 저장소의 벡터 차원 수 (임베딩 차원 수)
faiss_index.d

1024

In [25]:
# FAISS 벡터 저장소 생성
faiss_db = FAISS(
    embedding_function=embeddings_model,
    index=faiss_index,           # 벡터 검색을 위한 데이터 구조를 정의
    docstore=InMemoryDocstore(), # 문서 저장소 객체를 지정 - 문서의 원본 내용과 메타데이터를 보관
    index_to_docstore_id={},     # 인덱스와 문서 간의 연결을 관리 (매핑 딕셔너리)
)
# 저장된 문서의 갯수 확인
faiss_db.index.ntotal

0

`(2) 벡터 저장소 관리`  

- 문서 추가: `vector_store.add_documents(documents, ids)`

In [26]:
from langchain_core.documents import Document

# 문서 데이터 - (텍스트, 소스)
documents = [
    ("인공지능은 컴퓨터 과학의 한 분야입니다.", "AI 개론"),
    ("머신러닝은 인공지능의 하위 분야입니다.", "AI 개론"),
    ("딥러닝은 머신러닝의 한 종류입니다.", "딥러닝 입문"),
    ("자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.", "AI 개론"),
    ("컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.", "딥러닝 입문")
]

# Document 객체 생성
doc_objects = []
for content, source in documents:
    doc = Document(
        page_content=content,
        metadata={"source": source},
    )
    doc_objects.append(doc)


# 순차적 ID 리스트 생성
doc_ids = [f"DOC_{i}" for i in range(1, len(doc_objects) + 1)]

# 문서를 벡터 저장소에 저장
added_doc_ids = faiss_db.add_documents(documents=doc_objects, ids=doc_ids)

# 벡터 저장소에 저장된 문서를 확인
print(f"{len(added_doc_ids)}개의 문서가 성공적으로 벡터 저장소에 추가되었습니다.")
print(added_doc_ids)

5개의 문서가 성공적으로 벡터 저장소에 추가되었습니다.
['DOC_1', 'DOC_2', 'DOC_3', 'DOC_4', 'DOC_5']


In [27]:
# 저장된 문서의 갯수 확인
faiss_db.index.ntotal

5

In [28]:
# 저장된 인덱스 확인
faiss_db.index_to_docstore_id

{0: 'DOC_1', 1: 'DOC_2', 2: 'DOC_3', 3: 'DOC_4', 4: 'DOC_5'}

In [29]:
# 저장된 문서 검색
faiss_db.docstore.search('DOC_1')

Document(id='DOC_1', metadata={'source': 'AI 개론'}, page_content='인공지능은 컴퓨터 과학의 한 분야입니다.')

- 문서 삭제: `vector_store.delete(ids)`

In [30]:
# 문서 id를 지정하여 삭제
faiss_db.delete(ids=["DOC_5"])

True

In [31]:
# 컬렉션 확인
faiss_db.index.ntotal

4

In [32]:
# 저장된 인덱스 확인
faiss_db.index_to_docstore_id

{0: 'DOC_1', 1: 'DOC_2', 2: 'DOC_3', 3: 'DOC_4'}

In [33]:
# 저장된 문서 객체를 확인
faiss_db.docstore._dict

{'DOC_1': Document(id='DOC_1', metadata={'source': 'AI 개론'}, page_content='인공지능은 컴퓨터 과학의 한 분야입니다.'),
 'DOC_2': Document(id='DOC_2', metadata={'source': 'AI 개론'}, page_content='머신러닝은 인공지능의 하위 분야입니다.'),
 'DOC_3': Document(id='DOC_3', metadata={'source': '딥러닝 입문'}, page_content='딥러닝은 머신러닝의 한 종류입니다.'),
 'DOC_4': Document(id='DOC_4', metadata={'source': 'AI 개론'}, page_content='자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.')}

`(3) 문서 검색`  

- 유사도 검색
    - 주어진 쿼리와 가장 유사한 문서를 반환
    - k=2는 상위 2개의 결과를 반환하도록 지정
    - filter를 사용하여 특정 출처의 문서만 검색 가능

In [34]:
query = "인공지능과 머신러닝의 차이점은 무엇인가요?"
results = faiss_db.similarity_search(
    query,
    k=2,
    filter={"source": "AI 개론"}
)

print("유사도 검색 결과:")
for doc in results:
    print(f"- {doc.page_content} [출처: {doc.metadata['source']}]")

유사도 검색 결과:
- 머신러닝은 인공지능의 하위 분야입니다. [출처: AI 개론]
- 인공지능은 컴퓨터 과학의 한 분야입니다. [출처: AI 개론]


- 유사도 점수가 포함된 검색
    - 유사도 점수를 함께 반환
    - 점수가 낮을수록 더 유사한 것을 의미

In [35]:
query = "딥러닝은 어떤 분야에서 사용되나요?"
results = faiss_db.similarity_search_with_score(
    query,
    k=2,
    filter={"source": "AI 개론"}
)

print("점수가 포함된 유사도 검색 결과:\n")
for doc, score in results:
    print(f"- 점수: {score:.4f}")
    print(f"  내용: {doc.page_content}")
    print(f"  [출처: {doc.metadata['source']}]")
    print()

점수가 포함된 유사도 검색 결과:

- 점수: 0.8442
  내용: 머신러닝은 인공지능의 하위 분야입니다.
  [출처: AI 개론]

- 점수: 0.9845
  내용: 인공지능은 컴퓨터 과학의 한 분야입니다.
  [출처: AI 개론]



- 관련성 점수가 포함된 검색
    - 문서와 함께 0에서 1 사이의 관련성 점수를 반환
    - 0은 가장 관련성이 낮고, 1은 가장 관련성이 높음을 의미

In [36]:
query = "딥러닝은 어떤 분야에서 사용되나요?"
results = faiss_db.similarity_search_with_relevance_scores(
    query,
    k=2,
    filter={"source": "AI 개론"}
)

print(f"쿼리: {query}")
print("\n검색 결과 (관련성 점수 포함):")
for doc, score in results:
    print(f"- 관련성 점수: {score:.4f}")
    print(f"  내용: {doc.page_content}")
    print(f"  [출처: {doc.metadata['source']}]")
    print()

쿼리: 딥러닝은 어떤 분야에서 사용되나요?

검색 결과 (관련성 점수 포함):
- 관련성 점수: 0.4031
  내용: 머신러닝은 인공지능의 하위 분야입니다.
  [출처: AI 개론]

- 관련성 점수: 0.3038
  내용: 인공지능은 컴퓨터 과학의 한 분야입니다.
  [출처: AI 개론]



`(4) 로컬에 저장 및 로드`  

In [37]:
# 로컬에 저장
faiss_db.save_local("faiss_ai_sample_index")

In [38]:
# 로컬에 저장된 FAISS 벡터 저장소 불러오기
faiss_db2 = FAISS.load_local(
    "faiss_ai_sample_index", embeddings_model, allow_dangerous_deserialization=True
)

In [39]:
query = "딥러닝은 어떤 분야에서 사용되나요?"
results = faiss_db2.similarity_search_with_relevance_scores(
    query,
    k=2,
    filter={"source": "딥러닝 입문"}
)

print(f"쿼리: {query}")
print("\n검색 결과 (관련성 점수 포함):")
for doc, score in results:
    print(f"- 관련성 점수: {score:.4f}")
    print(f"  내용: {doc.page_content}")
    print(f"  [출처: {doc.metadata['source']}]")
    print()

쿼리: 딥러닝은 어떤 분야에서 사용되나요?

검색 결과 (관련성 점수 포함):
- 관련성 점수: 0.5392
  내용: 딥러닝은 머신러닝의 한 종류입니다.
  [출처: 딥러닝 입문]



### 3. **Pinecone**

- 상용 클라우드 기반의 벡터 데이터베이스 서비스 (SaaS)
- 실시간 고성능 벡터 검색 제공
- 회원가입 및 API 인증키 발급 (.env 파일에 `PINECONE_API_KEY` 등록)
- `langchain-pinecone pinecone-notebooks` 패키지 설치

`(1) 환경 설정`

In [40]:
# PINECONE_API_KEY 환경 변수 설정 후에 실행
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [41]:
# 인증 설정
import os
from pinecone import Pinecone, ServerlessSpec 
pinecone_api_key = os.environ.get("PINECONE_API_KEY")
if not pinecone_api_key:
    raise ValueError("PINECONE_API_KEY가 .env 파일에 설정되지 않았습니다.")
pc = Pinecone(api_key=pinecone_api_key)

In [42]:
pc.list_indexes()

[]

In [44]:
# 기존 인덱스 리스트 확인
existing_indexes = [index_info["name"] for index_info in pc.list_indexes()]
print(f"기존 인덱스: {existing_indexes}")

기존 인덱스: ['test']


`(2) 벡터 저장소 초기화`

In [45]:
import time 

# 인덱스 이름 설정
index_name = "ai-sample-index"

# 인덱스가 없으면 생성
if index_name not in existing_indexes:
    pc.create_index(
        name=index_name,
        dimension=1024,
        metric="euclidean",  # 유사도 측정 방법 - euclidean, cosine, dotproduct
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    while not pc.describe_index(index_name).status["ready"]:
        time.sleep(1)

# 인덱스 이름을 사용하여 인덱스 객체 생성
index = pc.Index(index_name)

In [46]:
index

In [47]:
# 인덱스 정보 확인
index_name = "ai-sample-index"

index_info = pc.describe_index(index_name)
index_info

{
    "name": "ai-sample-index",
    "metric": "euclidean",
    "host": "ai-sample-index-b8h8iut.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 1024,
    "deletion_protection": "disabled",
    "tags": null
}

In [48]:
from langchain_pinecone import PineconeVectorStore  

# PINECONE 벡터 저장소 생성
pinecone_db = PineconeVectorStore(index=index, embedding=embeddings_model)

# 벡터 저장소 객체 확인 
pinecone_db

In [49]:
# 저장된 문서의 갯수 확인
pinecone_db._index.describe_index_stats()

{'dimension': 1024,
 'index_fullness': 0.0,
 'metric': 'euclidean',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}

`(2) 벡터 저장소 관리`  

- 문서 추가: `vector_store.add_documents(documents, ids)`

In [50]:
from langchain_core.documents import Document

# 문서 데이터 - (텍스트, 소스)
documents = [
    ("인공지능은 컴퓨터 과학의 한 분야입니다.", "AI 개론"),
    ("머신러닝은 인공지능의 하위 분야입니다.", "AI 개론"),
    ("딥러닝은 머신러닝의 한 종류입니다.", "딥러닝 입문"),
    ("자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.", "AI 개론"),
    ("컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.", "딥러닝 입문")
]

# Document 객체 생성
doc_objects = []
for content, source in documents:
    doc = Document(
        page_content=content,
        metadata={"source": source},
    )
    doc_objects.append(doc)


# 순차적 ID 리스트 생성
doc_ids = [f"DOC_{i}" for i in range(1, len(doc_objects) + 1)]

# 문서를 벡터 저장소에 저장
added_doc_ids = pinecone_db.add_documents(documents=doc_objects, ids=doc_ids)

# 벡터 저장소에 저장된 문서를 확인
print(f"{len(added_doc_ids)}개의 문서가 성공적으로 벡터 저장소에 추가되었습니다.")
print(added_doc_ids)

5개의 문서가 성공적으로 벡터 저장소에 추가되었습니다.
['DOC_1', 'DOC_2', 'DOC_3', 'DOC_4', 'DOC_5']


In [51]:
# 저장된 문서의 갯수 확인 - 동기화에 시간이 걸릴 수 있음
pinecone_db._index.describe_index_stats()

{'dimension': 1024,
 'index_fullness': 0.0,
 'metric': 'euclidean',
 'namespaces': {'': {'vector_count': 5}},
 'total_vector_count': 5,
 'vector_type': 'dense'}

In [52]:
# 저장된 문서 검색
query = "인공지능과 머신러닝의 관계는?"
results = pinecone_db.similarity_search(query, k=2)

print(f"\n쿼리: {query}")
print("가장 유사한 문서:")
for doc in results:
    print(f"- {doc.page_content} [출처: {doc.metadata['source']}]")


쿼리: 인공지능과 머신러닝의 관계는?
가장 유사한 문서:
- 머신러닝은 인공지능의 하위 분야입니다. [출처: AI 개론]
- 딥러닝은 머신러닝의 한 종류입니다. [출처: 딥러닝 입문]


- 문서 삭제: `vector_store.delete(ids)`

In [53]:
# 문서 id를 지정하여 삭제
pinecone_db.delete(ids=["DOC_5"])

In [54]:
# 컬렉션 확인
pinecone_db._index.describe_index_stats()

{'dimension': 1024,
 'index_fullness': 0.0,
 'metric': 'euclidean',
 'namespaces': {'': {'vector_count': 4}},
 'total_vector_count': 4,
 'vector_type': 'dense'}

`(3) 문서 검색`  

- 유사도 검색
    - 주어진 쿼리와 가장 유사한 문서를 반환
    -  k=2는 상위 2개의 결과를 반환하도록 지정
    - filter를 사용하여 특정 출처의 문서만 검색 가능

In [55]:
query = "인공지능과 머신러닝의 차이점은 무엇인가요?"
results = pinecone_db.similarity_search(
    query,
    k=2,
    filter={"source": "AI 개론"}
)

print("유사도 검색 결과:")
for doc in results:
    print(f"- {doc.page_content} [출처: {doc.metadata['source']}]")

유사도 검색 결과:
- 머신러닝은 인공지능의 하위 분야입니다. [출처: AI 개론]
- 인공지능은 컴퓨터 과학의 한 분야입니다. [출처: AI 개론]


- 유사도 점수가 포함된 검색
    - 유사도 점수를 함께 반환
    - 점수가 낮을수록 더 유사한 것을 의미 (거리 기준으로 점수가 산정되기 때문)

In [56]:
query = "딥러닝은 어떤 분야에서 사용되나요?"
results = pinecone_db.similarity_search_with_score(
    query,
    k=2,
    filter={"source": "AI 개론"}
)

print("점수가 포함된 유사도 검색 결과:\n")
for doc, score in results:
    print(f"- 점수: {score:.4f}")
    print(f"  내용: {doc.page_content}")
    print(f"  [출처: {doc.metadata['source']}]")
    print()

점수가 포함된 유사도 검색 결과:

- 점수: 0.8432
  내용: 머신러닝은 인공지능의 하위 분야입니다.
  [출처: AI 개론]

- 점수: 0.9843
  내용: 인공지능은 컴퓨터 과학의 한 분야입니다.
  [출처: AI 개론]



- 관련성 점수가 포함된 검색
    - 문서와 함께 0에서 1 사이의 관련성 점수를 반환
    - 0은 가장 관련성이 낮고, 1은 가장 관련성이 높음을 의미

In [57]:
query = "딥러닝은 어떤 분야에서 사용되나요?"
results = pinecone_db.similarity_search_with_relevance_scores(
    query,
    k=2,
    filter={"source": "AI 개론"}
)

print(f"쿼리: {query}")
print("\n검색 결과 (관련성 점수 포함):")
for doc, score in results:
    print(f"- 관련성 점수: {score:.4f}")
    print(f"  내용: {doc.page_content}")
    print(f"  [출처: {doc.metadata['source']}]")
    print()

쿼리: 딥러닝은 어떤 분야에서 사용되나요?

검색 결과 (관련성 점수 포함):
- 관련성 점수: 0.9216
  내용: 머신러닝은 인공지능의 하위 분야입니다.
  [출처: AI 개론]

- 관련성 점수: 0.9922
  내용: 인공지능은 컴퓨터 과학의 한 분야입니다.
  [출처: AI 개론]



`(4) 벡터 저장소 로드`  

In [58]:
# 저장된 인덱스 확인해서 초기화 
index_name = "ai-sample-index"
index = pc.Index(index_name)
pinecone_db2 = PineconeVectorStore(index=index, embedding=embeddings_model)

# 저장된 문서 정보를 확인
pinecone_db2._index.describe_index_stats()

{'dimension': 1024,
 'index_fullness': 0.0,
 'metric': 'euclidean',
 'namespaces': {'': {'vector_count': 4}},
 'total_vector_count': 4,
 'vector_type': 'dense'}

In [59]:
# 미리 임베딩된 쿼리 벡터를 사용하여 검색
query = "딥러닝은 어떤 분야에서 사용되나요?"
results = pinecone_db2.similarity_search_with_relevance_scores(
    query,
    k=2,
    filter={"source": "AI 개론"}
)

print(f"쿼리: {query}")
print("\n검색 결과 (관련성 점수 포함):")
for doc, score in results:
    print(f"- 관련성 점수: {score:.4f}")
    print(f"  내용: {doc.page_content}")
    print(f"  [출처: {doc.metadata['source']}]")
    print()

쿼리: 딥러닝은 어떤 분야에서 사용되나요?

검색 결과 (관련성 점수 포함):
- 관련성 점수: 0.9216
  내용: 머신러닝은 인공지능의 하위 분야입니다.
  [출처: AI 개론]

- 관련성 점수: 0.9922
  내용: 인공지능은 컴퓨터 과학의 한 분야입니다.
  [출처: AI 개론]



# [실습 프로젝트]

### 실습 목표

1. 아래 샘플 문서들을 벡터 저장소에 저장하는 코드를 작성합니다. 
   - 적절한 벡터 저장소 선택 
   - 임베딩 모델 설정
   - 문서 구조 설계 (metadata 정의)

2. 벡터 저장소를 사용하여 다음 기능을 구현합니다. 
   - 새로운 문서 추가
   - 문서 삭제
   - 문서 검색: 유사도 점수 계산, 메타데이터 기반 필터링 등 

### 실습 단계

- **1단계: 벡터 저장소 초기화**
   - Chroma, FAISS, Pinecone 중 하나를 선택하고 이유를 설명하세요

- **2단계: 문서 저장**
   - 제공된 샘플 문서를 Document 객체로 변환하세요
   - metadata 구조를 설계하세요 (type, author 포함)

- **3단계: 문서 관리**
   - 새로운 문서 1개를 추가하세요
   - 특정 문서 1개를 삭제하세요

- **4단계: 문서 검색 구현**
   - [ ] 기본 유사도 검색
   - [ ] 메타데이터 필터링
   - [ ] 점수 포함 검색

In [13]:
# 샘플 문서 데이터 
documents = [
    {"content": "인공지능 기술의 발전과 미래", "type": "article", "author": "김철수"},
    {"content": "데이터 분석 입문 가이드", "type": "tutorial", "author": "이영희"},
    {"content": "머신러닝 모델 성능 개선 방법", "type": "research", "author": "박지성"},
    {"content": "블록체인 기술과 금융 혁신", "type": "article", "author": "정민우"},
    {"content": "클라우드 컴퓨팅 아키텍처 설계", "type": "tutorial", "author": "강다은"},
    {"content": "사이버 보안 위협 대응 전략", "type": "research", "author": "홍길동"},
    {"content": "빅데이터 처리 시스템 구축 사례", "type": "article", "author": "송지원"},
    {"content": "웹 개발자를 위한 REST API 가이드", "type": "tutorial", "author": "임성준"},
    {"content": "자연어 처리 알고리즘 비교 연구", "type": "research", "author": "최유진"},
    {"content": "디지털 트랜스포메이션 성공 전략", "type": "article", "author": "백승호"},
    {"content": "파이썬으로 시작하는 데이터 시각화", "type": "tutorial", "author": "유미란"},
    {"content": "강화학습을 활용한 게임 AI 개발", "type": "research", "author": "조현우"},
    {"content": "5G 네트워크 기술 동향", "type": "article", "author": "윤서연"},
    {"content": "도커 컨테이너 실전 가이드", "type": "tutorial", "author": "장민석"},
    {"content": "추천 시스템 최적화 연구", "type": "research", "author": "신영수"},
    {"content": "스마트 시티 구현 기술", "type": "article", "author": "권태영"},
    {"content": "깃허브 활용 협업 가이드", "type": "tutorial", "author": "오지훈"},
    {"content": "컴퓨터 비전 응용 사례 연구", "type": "research", "author": "남궁민"},
    {"content": "양자 컴퓨팅의 현재와 미래", "type": "article", "author": "하은주"},
    {"content": "리액트 네이티브 앱 개발 입문", "type": "tutorial", "author": "문동현"},
    {"content": "음성인식 시스템 성능 평가", "type": "research", "author": "심준호"},
    {"content": "메타버스 플랫폼 개발 동향", "type": "article", "author": "류아린"},
    {"content": "NoSQL 데이터베이스 설계 패턴", "type": "tutorial", "author": "반승현"},
    {"content": "엣지 컴퓨팅 적용 사례 연구", "type": "research", "author": "주민정"},
    {"content": "디지털 헬스케어 기술 혁신", "type": "article", "author": "구본우"},
    {"content": "마이크로서비스 아키텍처 구현", "type": "tutorial", "author": "염지현"},
    {"content": "강화학습 기반 로봇 제어 연구", "type": "research", "author": "탁현우"},
    {"content": "친환경 IT 인프라 구축 방안", "type": "article", "author": "방승미"},
    {"content": "프론트엔드 성능 최적화 기법", "type": "tutorial", "author": "곽준영"},
    {"content": "시계열 데이터 예측 모델 연구", "type": "research", "author": "추민서"}
]

In [14]:
# Document 객체 생성
from langchain_core.documents import Document

doc_objects = []

for i, doc in enumerate(documents): 
    doc_obj = Document(
        page_content=doc["content"],
        metadata={"type": doc["type"], "author": doc["author"]},
    )
    doc_objects.append(doc_obj)

# uuid 생성
import uuid

doc_ids = [str(uuid.uuid4()) for _ in range(len(doc_objects))]

### 여기에 나머지 코드를 작성하세요 ###

print("doc_objects 페이지 내용 :: " + doc_objects[0].page_content)
print("doc_objects 문서 :: {} ", len(doc_objects))
print("doc_ids[0] :: " + doc_ids[0])

doc_objects 페이지 내용 :: 인공지능 기술의 발전과 미래
doc_objects 문서 :: {}  30
doc_ids[0] :: 70d65afd-3db9-4a4d-9b09-1fcf9e14a276


# **1단계: 벡터 저장소 초기화**
- Chroma, FAISS, Pinecone 중 하나를 선택하고 이유를 설명하세요

Chroma 
https://docs.trychroma.com/docs

선정 이유 : 로컬 개발 환경에 최적화된 벡터 저장소를 선택 

벡터 저장소를 Chroma로 기본 학습 이후 다른 벡터 저장소도 적용

In [3]:
# 벡터 저장소에 문서를 저장할 때 적용할 임베딩 모델
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embeddings_model = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

e:\sw\dev\ai\modu_llm7\faq_bot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 30772.76it/s]


In [4]:
from langchain_chroma import Chroma

chroma_db = Chroma(
    collection_name="ai_practice_collection", # RDBMS로 치면  테이블
    embedding_function=embeddings_model,    # 필요한 임베딩 모델을 설정 
    persist_directory="./chroma_db",        # 저장할 폴더 경로
)

In [19]:
# 현재 저장된 컬렉션 데이터 확인
chroma_db.get()

{'ids': [],
 'embeddings': None,
 'documents': [],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': []}

- **2단계: 문서 저장**
   - 제공된 샘플 문서를 Document 객체로 변환하세요
   - metadata 구조를 설계하세요 (type, author 포함)

In [20]:
from langchain_core.documents import Document

# 문서를 벡터 저장소에 저장
added_doc_ids = chroma_db.add_documents(documents=doc_objects, ids=doc_ids)  # 문서 객체와 문서 id 입력 

# 벡터 저장소에 저장된 문서를 확인
print(f"{len(added_doc_ids)}개의 문서가 성공적으로 벡터 저장소에 추가되었습니다.")
print(added_doc_ids)

30개의 문서가 성공적으로 벡터 저장소에 추가되었습니다.
['0c18dee6-28b0-480d-a8e0-00eb1bf960fd', 'edb156a7-9086-4294-a624-0d4207722181', '47f8874f-1096-4abc-8540-2bcccf58c0b4', '05a1e2eb-2dca-4007-ab38-8aee21bf0605', '62638e34-c5cd-472f-ab8f-c257474d5c32', '8d648893-ddb5-478d-ad12-57c1f1aece66', '173020e7-a6d0-4c1a-a482-5852561e533b', '0816f852-9683-46b7-b702-cbc734da3745', '6ff2c071-ea5e-4ddc-b517-2c33e6f29028', 'dd393241-a3ae-4192-9147-5dd8e4e68310', 'ca210d53-cb94-42ed-94a0-a09057be0f17', '4081bf75-4fa2-42da-ba91-cea6bb01d89c', 'c49af3ea-d044-42d3-8d33-17de7349947c', '52cd059b-649b-4cd1-a1f9-b01493a871aa', 'ee11fdae-0779-4bd5-8f5d-6b29b9175019', '2665842e-9729-4688-b1e5-fad0bebb79af', '3a16fb40-d904-40bb-93b5-d002b0876a14', '9dafee1b-5161-4386-b459-ffb646fb4051', '26892bde-2c24-4ef6-86e5-dd147803e6b7', 'fc8eb282-cb36-447b-80d9-281277438b38', 'b90b5f44-aba6-448b-ab47-7780892cd26d', 'f6161d00-d4ff-4fde-9c41-a9d1dab3658c', '49928116-6a77-4a4d-a142-c40f09ceb366', '981711bb-f7dd-436c-b613-04e06b4812cf', '8a2a3f

- **3단계: 문서 관리**
   - 새로운 문서 1개를 추가하세요
   - 특정 문서 1개를 삭제하세요

In [23]:
# 새 문서 데이터 1건 
new_document = {"content": "데이터 아키텍처의 이해", "type": "article", "author": "박민수"}

# LangChain Document 추가
from langchain_core.documents import Document

new_doc_obj = Document(
        page_content=new_document["content"],
        metadata={"type": new_document["type"], "author": new_document["author"]},
    )

doc_objects.append(new_doc_obj)

# uuid 생성
import uuid

new_doc_id =  str(uuid.uuid4())


added_new_doc_ids = chroma_db.add_documents(documents=[new_doc_obj], ids=[new_doc_id])  # 문서 객체와 문서 id 입력 

# 벡터 저장소에 저장된 문서를 확인
print(f"{len(added_new_doc_ids)}개의 문서가 성공적으로 벡터 저장소에 추가되었습니다.")
print(added_new_doc_ids)

1개의 문서가 성공적으로 벡터 저장소에 추가되었습니다.
['36b8ae8e-9499-42b2-8725-6e1d3cc0f06a']


In [24]:
# 삭제할 특정 문서 ID 조회 
chroma_db.get()

{'ids': ['0c18dee6-28b0-480d-a8e0-00eb1bf960fd',
  'edb156a7-9086-4294-a624-0d4207722181',
  '47f8874f-1096-4abc-8540-2bcccf58c0b4',
  '05a1e2eb-2dca-4007-ab38-8aee21bf0605',
  '62638e34-c5cd-472f-ab8f-c257474d5c32',
  '8d648893-ddb5-478d-ad12-57c1f1aece66',
  '173020e7-a6d0-4c1a-a482-5852561e533b',
  '0816f852-9683-46b7-b702-cbc734da3745',
  '6ff2c071-ea5e-4ddc-b517-2c33e6f29028',
  'dd393241-a3ae-4192-9147-5dd8e4e68310',
  'ca210d53-cb94-42ed-94a0-a09057be0f17',
  '4081bf75-4fa2-42da-ba91-cea6bb01d89c',
  'c49af3ea-d044-42d3-8d33-17de7349947c',
  '52cd059b-649b-4cd1-a1f9-b01493a871aa',
  'ee11fdae-0779-4bd5-8f5d-6b29b9175019',
  '2665842e-9729-4688-b1e5-fad0bebb79af',
  '3a16fb40-d904-40bb-93b5-d002b0876a14',
  '9dafee1b-5161-4386-b459-ffb646fb4051',
  '26892bde-2c24-4ef6-86e5-dd147803e6b7',
  'fc8eb282-cb36-447b-80d9-281277438b38',
  'b90b5f44-aba6-448b-ab47-7780892cd26d',
  'f6161d00-d4ff-4fde-9c41-a9d1dab3658c',
  '49928116-6a77-4a4d-a142-c40f09ceb366',
  '981711bb-f7dd-436c-b613-

In [27]:
# 삭제할 문서 id 조회 
print("조회한 문서 :: {} ", chroma_db.get_by_ids(['49928116-6a77-4a4d-a142-c40f09ceb366']));

# 특정 문서 id를 지정하여 삭제
chroma_db.delete(ids=["49928116-6a77-4a4d-a142-c40f09ceb366"])

# 삭제된 문서 id 조회
print("삭제된 ID '49928116-6a77-4a4d-a142-c40f09ceb366' 문서 조회 :: {}" ,  chroma_db.get_by_ids(['49928116-6a77-4a4d-a142-c40f09ceb366']));

조회한 문서 :: {}  [Document(id='49928116-6a77-4a4d-a142-c40f09ceb366', metadata={'author': '반승현', 'type': 'tutorial'}, page_content='NoSQL 데이터베이스 설계 패턴')]
삭제된 ID '49928116-6a77-4a4d-a142-c40f09ceb366' 문서 조회 :: {} []


- **4단계: 문서 검색 구현**
   - [ ] 기본 유사도 검색
   - [ ] 메타데이터 필터링
   - [ ] 점수 포함 검색

In [35]:
query = "데이터 거버넌스는 기업의 데이터 자산을 관리하고 통제하기 위한 정책"

results = chroma_db.similarity_search_with_score(
    query,
    k=2,
    filter={"type": "article"}
)

print("점수가 포함된 유사도 검색 결과:\n")
for doc, score in results:
    print(f"- 점수: {score:.4f}")
    print(f"  내용: {doc.page_content}")
    print(f"  [출처: {doc.metadata['type']} , {doc.metadata['author']}]")
    print()

점수가 포함된 유사도 검색 결과:

- 점수: 0.8553
  내용: 데이터 아키텍처의 이해
  [출처: article , 박민수]

- 점수: 1.0439
  내용: 디지털 트랜스포메이션 성공 전략
  [출처: article , 백승호]



In [1]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embeddings_model = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

d:\developer\ai\edu\modu_llm7\faq_bot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\developer\ai\edu\modu_llm7\faq_bot\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in D:\developer\ai\hf_cache\hub\models--BAAI--bge-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate develop

### Qdrant 


In [2]:
from qdrant_client import QdrantClient

client = QdrantClient(url="http://172.17.0.51:6333")


In [3]:
from qdrant_client.models import Distance, VectorParams

client.create_collection(
    collection_name="test_collection",
    vectors_config=VectorParams(size=4, distance=Distance.DOT),
)

True

In [4]:
from qdrant_client.models import PointStruct

operation_info = client.upsert(
    collection_name="test_collection",
    wait=True,
    points=[
        PointStruct(id=1, vector=[0.05, 0.61, 0.76, 0.74], payload={"city": "Berlin"}),
        PointStruct(id=2, vector=[0.19, 0.81, 0.75, 0.11], payload={"city": "London"}),
        PointStruct(id=3, vector=[0.36, 0.55, 0.47, 0.94], payload={"city": "Moscow"}),
        PointStruct(id=4, vector=[0.18, 0.01, 0.85, 0.80], payload={"city": "New York"}),
        PointStruct(id=5, vector=[0.24, 0.18, 0.22, 0.44], payload={"city": "Beijing"}),
        PointStruct(id=6, vector=[0.35, 0.08, 0.11, 0.44], payload={"city": "Mumbai"}),
    ],
)

print(operation_info)

operation_id=1 status=<UpdateStatus.COMPLETED: 'completed'>


In [5]:
search_result = client.query_points(
    collection_name="test_collection",
    query=[0.2, 0.1, 0.9, 0.7],
    with_payload=False,
    limit=3
).points

print(search_result)

[ScoredPoint(id=4, version=1, score=1.362, payload=None, vector=None, shard_key=None, order_value=None), ScoredPoint(id=1, version=1, score=1.273, payload=None, vector=None, shard_key=None, order_value=None), ScoredPoint(id=3, version=1, score=1.208, payload=None, vector=None, shard_key=None, order_value=None)]


In [6]:
from qdrant_client.models import Filter, FieldCondition, MatchValue

search_result = client.query_points(
    collection_name="test_collection",
    query=[0.2, 0.1, 0.9, 0.7],
    query_filter=Filter(
        must=[FieldCondition(key="city", match=MatchValue(value="London"))]
    ),
    with_payload=True,
    limit=3,
).points

print(search_result)

[ScoredPoint(id=2, version=1, score=0.871, payload={'city': 'London'}, vector=None, shard_key=None, order_value=None)]


In [12]:
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient

# 1. 임베딩 및 Qdrant 클라이언트 설정
client = QdrantClient(url="http://172.17.0.51:6333")

vector_size = len(embeddings_model.embed_query("sample text"))

print(vector_size)

1024


In [19]:
client.create_collection(
    collection_name="my_collection2",
    vectors_config=VectorParams(size=1024, distance=Distance.COSINE),
)

# 2. Qdrant 벡터 저장소 객체 생성
qdrant = QdrantVectorStore(
    client=client,
    collection_name="my_collection2",
    embedding=embeddings_model
)


# 문서 데이터 추가

In [20]:
from langchain_core.documents import Document

# 문서를 벡터 저장소에 저장
added_doc_ids = qdrant.add_documents(documents=doc_objects, ids=doc_ids)  # 문서 객체와 문서 id 입력 

# 벡터 저장소에 저장된 문서를 확인
print(f"{len(added_doc_ids)}개의 문서가 성공적으로 벡터 저장소에 추가되었습니다.")
print(added_doc_ids)

30개의 문서가 성공적으로 벡터 저장소에 추가되었습니다.
['70d65afd-3db9-4a4d-9b09-1fcf9e14a276', '440d67f7-a0c3-426a-b7d9-d67a3970671a', '6938082b-6cf9-4f62-890b-e501dfdd0791', '3f7bee33-3ef5-426c-be65-d15c452fa220', 'c670ee2f-fe7c-4654-8992-b853bdb9d542', '37e507c7-41bf-4a68-b18a-131d2afb85fd', '0d5df76f-f3b8-41d8-85d6-ef10600f5b87', 'a589b774-7066-41c1-ba99-f8b289d884ff', '2763c91f-fa9b-43e7-a090-e553813b0c90', 'e6c0ef45-1c08-434b-be7c-87199d011a4c', 'db289aee-e00e-4828-8ec4-6fafaa0332c4', 'bb693d8a-e82e-4718-a2f9-ba03f0d34867', 'c881b56f-8aa6-4645-8462-0066da795dc8', '2fe57335-5b06-4f39-90f2-132c45e9c59a', 'b8bf6996-32d0-4c3f-88d6-cb4948c1447f', '45f6152a-f028-41e2-8134-52d0bc967bf9', '7a645184-e765-4826-b610-49c3e16e7310', '56557bf1-0d29-4cab-b91f-abb1f8d36bee', '8e674299-1165-4f45-9f25-909a5ffa4b40', 'f7411588-8f33-43c6-af62-67cb59a04377', 'ee39796a-eab0-4556-bc3e-3774d60cca4e', '67db67e9-315f-4aa0-9e83-0b203557f6ca', '8e7c71b8-53f8-41e6-b038-383caedc256d', '0fbc54e7-8814-4c39-93e0-4a6e0c0969bc', 'bfb5f2

In [21]:
# 3. 유사도 및 점수 검색 실행
query = "데이터분석"
results = qdrant.similarity_search_with_score(query, k=3)

# 4. 결과 출력
for doc, score in results:
    print(f"유사도 점수: {score}")
    print(f"문서 내용: {doc.page_content}")
    print(f"메타데이터: {doc.metadata}\n")

유사도 점수: 0.66739833
문서 내용: 데이터 분석 입문 가이드
메타데이터: {'type': 'tutorial', 'author': '이영희', '_id': '440d67f7-a0c3-426a-b7d9-d67a3970671a', '_collection_name': 'my_collection2'}

유사도 점수: 0.6456484
문서 내용: 시계열 데이터 예측 모델 연구
메타데이터: {'type': 'research', 'author': '추민서', '_id': '9d624645-296e-4272-a203-0425ed0ecf32', '_collection_name': 'my_collection2'}

유사도 점수: 0.5760138
문서 내용: 빅데이터 처리 시스템 구축 사례
메타데이터: {'type': 'article', 'author': '송지원', '_id': '0d5df76f-f3b8-41d8-85d6-ef10600f5b87', '_collection_name': 'my_collection2'}

